# Tree of Thoughts and MCTS — interactive companion

Companion to [Post 3c: Tree of Thoughts and MCTS](../posts/03c-tree-of-thoughts.qmd).

A ReAct path is one trajectory; one bad step derails it. This notebook turns
reasoning into search over a tree and shows how backtracking and exploration
recover from the errors that compound for a single path.

**You'll do (~20 minutes):**
1. Build a reasoning tree and inspect its planted solutions.
2. Watch greedy fail where DFS and MCTS succeed.
3. See how each method holds up as the value heuristic gets noisier.
4. Reproduce the payoff: search mitigating error compounding.
5. Tune the MCTS exploration constant and watch the bandit tradeoff appear.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.search import (
    ReasoningTree, NoisyValueHeuristic, rollout_value,
    greedy_search, beam_search_tot, dfs_tot, mcts,
)

rng = np.random.default_rng(0)

## 1. A reasoning tree

Nodes are tuples of choices: `()` is the root, `(0, 2)` a grandchild, and
length-`depth` tuples are leaves. A few leaves are "correct" (reward 1),
clustered under a couple of subtrees to create a value gradient that search
can climb. The tree is small enough to enumerate exactly.

In [ ]:
tree = ReasoningTree(depth=4, branching=3, n_correct=4,
                     n_clusters=2, cluster_depth=2, seed=3)
print(f"depth={tree.depth}, branching={tree.branching}, "
      f"{tree.n_leaves} leaves")
print(f"correct leaf indices: {sorted(tree.correct_leaves)}")
print(f"root value (fraction of leaves correct): {tree.true_value(()):.3f}")

# Value of each top-level child — the gradient search will try to climb.
print("\nTop-level child values:")
for c in tree.children(()):
    print(f"  {c}: true value = {tree.true_value(c):.3f}")

The children have different values — the subtrees containing clustered
solutions score higher. A perfect heuristic would always descend into the
highest-value child. A noisy one might not.

### Try this
- Change `n_clusters=1` to concentrate all solutions in one subtree (sharper
  gradient). Or `n_clusters=4` to spread them out (flatter, harder).
- Change the `seed` for a different tree.

## 2. Greedy fails where search succeeds

With a noisy heuristic, greedy commits to one path and can't recover. DFS
backtracks; MCTS explores. Let's run all four on the same trees.

In [ ]:
def evaluate(method_fn, n_trees=200, noise=0.25, budget=64, **kw):
    n_correct = 0
    for t in range(n_trees):
        tr = ReasoningTree(depth=5, branching=3, n_correct=4,
                           n_clusters=2, cluster_depth=2, seed=t)
        h = NoisyValueHeuristic(tr, noise=noise, seed=t)
        found = method_fn(tr, h, budget, **kw)
        n_correct += int(found)
    return n_correct / n_trees

budget = 64
print(f"Success rate over 200 trees (noise=0.25, budget={budget}):\n")
print(f"  greedy:        {evaluate(lambda tr,h,b: greedy_search(tr,h,b)[0], budget=budget):.2f}")
print(f"  ToT-beam b=3:  {evaluate(lambda tr,h,b: beam_search_tot(tr,h,3,b)[0], budget=budget):.2f}")
print(f"  ToT-DFS:       {evaluate(lambda tr,h,b: dfs_tot(tr,h,b)[0], budget=budget):.2f}")
print(f"  MCTS:          {evaluate(lambda tr,h,b: mcts(tr,h,b,1.0,np.random.default_rng(0))[0], budget=budget):.2f}")

Greedy and fixed-beam plateau; DFS and MCTS use the budget to recover.
On these small trees, DFS is especially strong — exhaustive backtracking is
hard to beat when the tree is small enough to mostly cover.

### Try this
- Raise `budget` to 256. Which methods improve? (Greedy won't — it can't
  use extra budget without backtracking.)
- Lower it to 16. Now even DFS struggles.

## 3. Robustness to heuristic noise

The value heuristic is never perfect. Watch how each method degrades as we
crank up the noise, at a generous fixed budget.

In [ ]:
noises = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.5]
budget = 128
methods = {
    "greedy": lambda tr,h,b: greedy_search(tr,h,b)[0],
    "ToT-beam (b=3)": lambda tr,h,b: beam_search_tot(tr,h,3,b)[0],
    "ToT-DFS": lambda tr,h,b: dfs_tot(tr,h,b)[0],
    "MCTS": lambda tr,h,b: mcts(tr,h,b,1.0,np.random.default_rng(0))[0],
}
colors = {"greedy":"#888","ToT-beam (b=3)":"#3a7ebf","ToT-DFS":"#dd8452","MCTS":"#c44e52"}

for name, fn in methods.items():
    succ = [evaluate(fn, noise=nz, budget=budget) for nz in noises]
    plt.plot(noises, succ, "o-", color=colors[name], lw=2, label=name)
plt.xlabel("heuristic noise"); plt.ylabel("fraction solved")
plt.title(f"Noise robustness (budget={budget})"); plt.legend()
plt.grid(alpha=0.3); plt.ylim(-0.05, 1.05); plt.show()

DFS stays flat — backtracking makes it immune to a bad heuristic given enough
budget. MCTS is also robust (exploration recovers). Greedy and beam collapse:
no recovery mechanism. The methods that can *undo* a bad decision are the ones
that tolerate unreliable guidance.

### Try this
- Push the noise past 1.5. Does DFS ever break? (At a fixed budget on a fixed
  tree size, it stays robust because it eventually explores everything.)

## 4. The payoff: search mitigates error compounding

Post 3b showed a single path's success decays as $(1-p)^L$. Here greedy
(single path) decays with reasoning depth, while DFS recovers — until the
tree outgrows the fixed budget.

In [ ]:
depths = [2, 3, 4, 5, 6, 7]
noise, budget = 0.3, 256
for name, fn, color in [
    ("greedy (single path)", lambda tr,h,b: greedy_search(tr,h,b)[0], "#c44e52"),
    ("ToT-DFS (search)",     lambda tr,h,b: dfs_tot(tr,h,b)[0], "#dd8452"),
]:
    succ = []
    for depth in depths:
        n_correct = 0
        for t in range(120):
            tr = ReasoningTree(depth=depth, branching=3, n_correct=4,
                               n_clusters=2, cluster_depth=min(2, depth-1), seed=t)
            h = NoisyValueHeuristic(tr, noise=noise, seed=t)
            n_correct += int(fn(tr, h, budget))
        succ.append(n_correct / 120)
    plt.plot(depths, succ, "o-", color=color, lw=2, ms=9, label=name)
plt.xlabel("reasoning depth"); plt.ylabel("fraction solved")
plt.title(f"Search vs error compounding (noise={noise}, budget={budget})")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(-0.05, 1.05); plt.show()

The greedy single-path curve compounds toward zero. DFS holds high far longer —
but note it eventually declines too: with a fixed budget and a tree growing as
$3^{\rm depth}$, search can't fully outrun exponential growth. Search converts
compute into reliability; deeper reasoning just needs more of it.

### Try this
- Bump `budget` to 1024 and watch the DFS curve hold up to greater depth.

## 5. The MCTS exploration constant

UCT balances exploitation (node value) and exploration (visit-count bonus)
via `c_uct`. The best value depends on your budget — the bandit tradeoff from
Topic 1, now inside a tree.

In [ ]:
c_values = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
for budget, color in [(32, "#3a7ebf"), (128, "#c44e52")]:
    succ = []
    for c in c_values:
        n_correct = 0
        for t in range(150):
            tr = ReasoningTree(depth=5, branching=3, n_correct=4,
                               n_clusters=2, cluster_depth=2, seed=t)
            h = NoisyValueHeuristic(tr, noise=0.3, seed=t)
            n_correct += int(mcts(tr, h, budget, c, np.random.default_rng(t))[0])
        succ.append(n_correct / 150)
    plt.plot(c_values, succ, "o-", color=color, lw=2, ms=9, label=f"budget={budget}")
plt.xlabel("exploration constant c_uct"); plt.ylabel("fraction solved")
plt.title("MCTS exploration constant"); plt.legend()
plt.grid(alpha=0.3); plt.ylim(-0.05, 1.05); plt.show()

At low budget, `c=0` (pure exploitation) is best — you can't afford to explore.
At high budget, a bit of exploration (`c≈0.25-0.5`) wins. Exploration is an
investment that only pays off when you have the compute to spend on it.

### Try this
- Add `budget=512`. Does the sweet spot shift further toward exploration?

## What's next

You've built search over reasoning from scratch:

- **Tree of Thoughts** (BFS/DFS) — systematic exploration with a value heuristic.
- **MCTS with UCT** — bandit-style adaptive allocation, the AlphaZero engine.
- **Noise robustness** — backtracking and exploration as recovery mechanisms.
- **Error recovery** — search mitigating the $(1-p)^L$ compounding from Post 3b.

That completes Topic 3 — the full arc of inference-time computation, from
decoding to tools to search.

**Next up — Topic 4: Memory, retrieval, and reflection.** Agents over long
horizons must remember, retrieve, and evaluate their own outputs. We'll treat
retrieval as Bayesian conditioning, reflection as Monte Carlo quality
estimation, and tackle the calibration problem an agent needs to decide *when*
to retrieve, reflect, or act.